# EOT Benchmark — Results

Loads `.pkl` result files from `results/<problem>/` and saves comparison plots to `figures/`.

## How to use

1. Run the experiment from the terminal first:
   ```bash
   python main.py --problem gaussian
   ```
2. Set `PROBLEM` in the cell below to match what you ran.
3. Execute the notebook top-to-bottom. Plots are saved to `figures/`.

## What the plots show

- **Top panel:** gradient norm vs. number of iterations (adjust `max_iter` in the last cell)
- **Bottom panel:** gradient norm vs. wall-clock time — cutoff is read from `config.yaml` (`max_time`)

In [ ]:
import os
import pickle
import yaml
import numpy as np
import matplotlib.pyplot as plt

# ── Set this to match what you ran ────────────────────────────────────────────
PROBLEM = 'gaussian'   # choices: gaussian | gene | knowledge_distillation
# ──────────────────────────────────────────────────────────────────────────────

with open('config.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['problem']['type'] = PROBLEM

RESULTS_DIR = os.path.join(cfg['output']['results_dir'], PROBLEM)
FIGURES_DIR = cfg['output']['figures_dir']
os.makedirs(FIGURES_DIR, exist_ok=True)

In [ ]:
# rank_label: the parameter name to show in the legend, or None for no annotation
CONFIG = {
    'Sinkhorn':      {'label': 'Sinkhorn',               'color': 'b',      'marker': 'o', 'rank_label': None},
    'Newton':        {'label': 'Newton',                 'color': 'r',      'marker': 's', 'rank_label': None},
    'GRN':           {'label': 'GRN',                    'color': 'y',      'marker': 'x', 'rank_label': None},
    'SORN':          {'label': r'$\mathbf{SORN}$ (Ours)', 'color': 'g',     'marker': '^', 'rank_label': 'rank'},
    'Newton_Sketch': {'label': 'RSN',                    'color': 'm',      'marker': 'D', 'rank_label': 'sketch_dim'},
    'SGN':           {'label': 'SGN',                    'color': 'orange', 'marker': 'h', 'rank_label': 'sketch_dim'},
    'KCRN':          {'label': 'Krylov-CRN',             'color': 'c',      'marker': '*', 'rank_label': 'Krylov_dim'},
    'SSCN':          {'label': 'SSCN',                   'color': 'k',      'marker': 'p', 'rank_label': 'sketch_dim'},
    'AGD':           {'label': 'NAGD',                   'color': 'brown',  'marker': 'v', 'rank_label': None},
    'LBFGS':         {'label': 'L-BFGS',                 'color': 'teal',   'marker': '_', 'rank_label': None},
}

In [ ]:
def load_results(results_dir):
    """Load all .pkl files from results_dir. Returns {solver_name: metrics_dict}."""
    data = {}
    for fname in os.listdir(results_dir):
        if not fname.endswith('.pkl'):
            continue
        solver_name = fname.split('_')[0]
        fpath = os.path.join(results_dir, fname)
        with open(fpath, 'rb') as f:
            data[solver_name] = pickle.load(f)
        print(f'Loaded {solver_name} from {fname}')
    return data

results = load_results(RESULTS_DIR)

In [ ]:
def plot_comparison(results, cfg, max_iter=50, filename='comparison'):
    """
    Two-panel plot (stacked vertically):
      top    — gradient norm vs. iterations (first max_iter iterations)
      bottom — gradient norm vs. wall time (cutoff from config max_time)
    Solvers not present in results are silently skipped.
    """
    ptype    = cfg['problem']['type']
    rank     = cfg['problem'][ptype]['rank']
    max_time = cfg['problem'][ptype]['max_time']

    LABEL_SIZE = 18

    fig, axes = plt.subplots(2, 1, figsize=(8, 10))

    plot_order = sorted(results.keys(), key=lambda k: k == 'SORN')

    for method in plot_order:
        if method not in CONFIG:
            continue
        m    = results[method]
        cfg_ = CONFIG[method]

        # Build legend label: append (param=rank) for sketch-based methods
        label = cfg_['label']
        if cfg_['rank_label'] is not None:
            label += f" ({cfg_['rank_label']}={rank})"

        kw = dict(linestyle='-', color=cfg_['color'],
                  marker=cfg_['marker'], markersize=4, markevery=1,
                  label=label)

        errors = m.get('error', [])

        # Top: vs iterations
        n = min(max_iter, len(errors))
        if n > 0:
            axes[0].semilogy(range(n), errors[:n], **kw)

        # Bottom: vs time, cut at max_time
        times = m.get('time', [])
        cut = int(np.searchsorted(times, max_time))
        if cut > 0:
            axes[1].semilogy(times[:cut], errors[:cut], **kw)

    for ax, xlabel in zip(axes, ['Iteration', 'Time (s)']):
        ax.set_xlabel(xlabel, fontsize=LABEL_SIZE)
        ax.set_ylabel('Gradient norm', fontsize=LABEL_SIZE)
        ax.tick_params(labelsize=LABEL_SIZE - 2)
        ax.legend(fontsize=LABEL_SIZE - 4)
        ax.grid(True)

    plt.tight_layout(pad=0.5)
    out = os.path.join(FIGURES_DIR, f'{filename}.png')
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved → {out}')

In [ ]:
plot_comparison(results, cfg, max_iter=50, filename='main_comparison')